In [ ]:
import numpy as NP
from function import rosenbrock, plot_all_models, plot_rosenbrock, plot_obs, corner_plot
from config import type_banchmark
from MT.MT_function import forward
from tqdm import tqdm
from NA.Appraise import NAAppraiser
from NA.SEARCH import sampling_jit,generate_random_models,dim_models



nss= ns
type = int(input("1. rosenbrock, 2. MT1D..? "))  


lb, ub, frequencies, Obsres, Obsphs, ResReal, ThkReal , n , maxdepth= type_banchmark(type)

nd = lb.size
ne = ns * (iter - 1) + ni


while True:

    if type != 1:
        ERes = NP.zeros((ne,len(frequencies)))        
        EPhs = NP.zeros((ne,len(frequencies)))     

    models = NP.zeros((ne, nd))
    misfits = NP.zeros(ne)        

    np = 0       
    idx = 0
            
    for it in tqdm(range(iter), desc="Progress"):
        if it == 0:
            ns = ni 
            batch = generate_random_models(ni,nd)
            background = lb + (ub - lb) * generate_random_models(50000,nd)
            background_misfits = rosenbrock(background)

        else:
            ns = nss 
            batch = sampling_jit(ns,
                            nd,
                            nr,
                            models,
                            np,
                            misfits)                                 
        
        models[idx:idx+ns] = batch
        batch = lb + (ub - lb) * batch

        if type == 1:
            misfits[idx:idx+ns] = rosenbrock(batch)
        else:
            misfits[idx:idx+ns],ERes[idx:idx+ns],EPhs[idx:idx+ns] = forward(batch, frequencies, Obsres, Obsphs)

        idx += ns
        np += ns
       
    best_idx = NP.argmin(misfits)
    models = dim_models(lb, ub, models)
    best_model = models[best_idx]
    print("Best model:", best_model)
    print(f"Best misfit: {misfits[best_idx]:.3f}")

    if type == 1:
        plot_rosenbrock(models, best_model, background, background_misfits, save_path=f"Images/rosenbrock.png")

    else: 
        Resis = best_model[:n]  
        Thick = best_model[n:]
        bestmisfit, res, phs = forward(best_model.reshape(1,-1), frequencies, Obsres, Obsphs) 
        plot_obs(frequencies, Obsres, Obsphs, res, phs, ERes, EPhs,save_path=f"Images/curve_{n}layer.png")
        plot_all_models(models,Resis,Thick,ResReal,ThkReal,n,save_path=f"Images/model_{n}layer.png", depthmax=maxdepth)
        print(np)


    """if  misfits[best_idx]< 9:
        break"""

    repeat = input("Repeat program? (y/n): ")
    
    if repeat.lower() != 'y':
        break 


"""
RESAMPLE = int(input("jangan terlalu banyak lama karena core colab hanya 2, coba 1000-5000, N_RESAMPLE : ?? "))
results = NAAppraiser(
    initial_ensemble=models,
    log_ppd= -misfits,
    bounds = tuple(zip(lb, ub)),
    n_resample=RESAMPLE,
    n_walkers=2
)

results.run()


if type ==1 :
    true_model = NP.array([1,1])
else : true_model = NP.hstack((ResReal, ThkReal))


corner_plot(
    type=type,
    samples=results.samples,
    mean=results.mean,
    best_model=best_model,
    true_model=true_model,
)"""

MT 1D Function




Progress: 100%|██████████| 250/250 [00:16<00:00, 15.04it/s]


Best model: [169.82481872   6.61755571 962.2774491  346.31214304 381.32028513]
Best misfit: 3.416
25000


Progress: 100%|██████████| 250/250 [00:01<00:00, 133.33it/s]


Best model: [1807.41690678   11.69634423  985.62790778  254.37293984  708.34577063]
Best misfit: 2.730
25000


Progress: 100%|██████████| 250/250 [00:01<00:00, 136.03it/s]


Best model: [733.226908    11.72199491 989.16215635 259.07531068 711.00038809]
Best misfit: 2.579
25000


'\nRESAMPLE = int(input("jangan terlalu banyak lama karena core colab hanya 2, coba 1000-5000, N_RESAMPLE : ?? "))\nresults = NAAppraiser(\n    initial_ensemble=models,\n    log_ppd= -misfits,\n    bounds = tuple(zip(lb, ub)),\n    n_resample=RESAMPLE,\n    n_walkers=2\n)\n\nresults.run()\n\n\nif type ==1 :\n    true_model = NP.array([1,1])\nelse : true_model = NP.hstack((ResReal, ThkReal))\n\n\ncorner_plot(\n    type=type,\n    samples=results.samples,\n    mean=results.mean,\n    best_model=best_model,\n    true_model=true_model,\n)'

In [2]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(misfits, linewidth=0.2)
plt.xlabel('Model Index')
plt.ylabel('Misfit')
plt.title('Global Misfit Values')
plt.savefig("Images/misfit_plot.png", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
"""def plot_obs(
    frequencies,
    Obsres, Obsphs,
    res_best, phs_best,
    EsembleRes, EsemblePhs,
    save_path="curve.png"
):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection

    # ---- normalisasi shape ----
    frequencies = np.asarray(frequencies).squeeze()
    Obsres      = np.asarray(Obsres).squeeze()
    Obsphs      = np.asarray(Obsphs).squeeze()
    res_best    = np.asarray(res_best).squeeze()
    phs_best    = np.asarray(phs_best).squeeze()

    n_ens = EsembleRes.shape[0]

    # ---- bangun segmen garis (SEMUA ensemble) ----
    seg_res = [np.column_stack((frequencies, EsembleRes[i])) for i in range(n_ens)]
    seg_phs = [np.column_stack((frequencies, EsemblePhs[i])) for i in range(n_ens)]

    fig, axes = plt.subplots(2, 1, figsize=(8, 10), sharex=True)

    # ================= Resistivity =================
    ax = axes[0]
    lc_res = LineCollection(seg_res, colors="blue", alpha=0.01, linewidths=0.8)
    ax.add_collection(lc_res)

    ax.scatter(frequencies, Obsres, color="black", s=25, zorder=5, label="Observed")
    ax.plot(frequencies, res_best, color="red", linewidth=2, label="Best Invert")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylabel("Apparent Resistivity (Ω·m)")
    ax.set_title("Observed vs Inverted Resistivity")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    ax.autoscale()

    # ================= Phase =================
    ax = axes[1]
    lc_phs = LineCollection(seg_phs, colors="blue", alpha=0.01, linewidths=0.8)
    ax.add_collection(lc_phs)

    ax.scatter(frequencies, Obsphs, color="black", s=25, zorder=5, label="Observed")
    ax.plot(frequencies, phs_best, color="red", linewidth=2, label="Best Invert")

    ax.set_xscale("log")
    ax.set_ylabel("Phase (°)")
    ax.set_xlabel("Frequency (Hz)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    ax.autoscale()

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close(fig)

    return save_path
"""

In [19]:
"""plot_obs(frequencies, Obsres, Obsphs, res, phs, ERes, EPhs,save_path=f"curve_{n}layer.png")
        """

C:\Users\h y p e\AppData\Local\Temp\ipykernel_61804\2353024720.py:58: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
C:\Users\h y p e\AppData\Local\Temp\ipykernel_61804\2353024720.py:59: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(save_path, dpi=120, bbox_inches="tight")


'curve_4layer.png'